# 03 — Robustness Checks

Re-run the primary test under variations of nuisance parameters. **No
re-selection** based on robustness outcomes — these are sensitivity
analyses, not decision points.

Pre-registered variations (PRE_REGISTRATION.md §9):
- `tcost ∈ {0, 10, 20 (main), 40}` bps
- Re-estimation freq ∈ {annual (main), quarterly, triennial}
- Sub-period split ∈ {full (main), 1977–1999, 2000–2024}
- Asset/Port vol target ∈ {10%/10% (main), 10%/8%, 10%/12%}


In [1]:
import sys, os
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from trend import (fetch_ff49, fetch_tbill_1m, grid_backtest, walkforward_oos,
                    lw_sharpe_diff_bootstrap, summarize)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white"})
RESULTS = "../results"


In [2]:
ff49 = fetch_ff49(cache_dir="../data")
rf   = fetch_tbill_1m(cache_dir="../data")


## 1. Robustness over transaction cost

In [3]:
rows = []
for tc in [0, 10, 20, 40]:
    g = grid_backtest(ff49, rf, tcost_bps=tc)
    for proto, tw in [("Expanding", "expanding"), ("Rolling 30y", 30)]:
        oos, _ = walkforward_oos(g, warmup_end="1976-06-30", train_window=tw)
        bl = g.loc[oos.index, "fast=1,wf=+0.0"]
        out = lw_sharpe_diff_bootstrap(oos, bl, mean_block_size=12, n_boot=5000, seed=42, alternative="greater")
        rows.append({"tcost_bps": tc, "protocol": proto,
                     "SR_best": out["sr_a"], "SR_base": out["sr_b"],
                     "delta": out["delta"], "p_value": out["p_value"]})
df_tc = pd.DataFrame(rows)
display(df_tc.round(4))
df_tc.to_csv(f"{RESULTS}/03_robustness_tcost.csv", index=False)


,tcost_bps,protocol,SR_best,SR_base,delta,p_value
0,0,Expanding,0.4767,0.3521,0.1246,0.0996
1,0,Rolling 30y,0.4531,0.3521,0.1010,0.1296
2,10,Expanding,0.4300,0.3378,0.0922,0.1698
3,10,Rolling 30y,0.3980,0.3378,0.0602,0.2514
4,20,Expanding,0.3866,0.3235,0.0631,0.2530
5,20,Rolling 30y,0.3634,0.3235,0.0400,0.3218
6,40,Expanding,0.2722,0.2948,-0.0226,0.5954
7,40,Rolling 30y,0.3113,0.2948,0.0165,0.4080


## 2. Robustness over sub-periods

In [4]:
g = grid_backtest(ff49, rf, tcost_bps=20.0)
rows = []
for proto, tw in [("Expanding", "expanding"), ("Rolling 30y", 30)]:
    oos, _ = walkforward_oos(g, warmup_end="1976-06-30", train_window=tw)
    bl = g.loc[oos.index, "fast=1,wf=+0.0"]
    for sub_name, sub_slice in [("Full", slice("1976-07", None)),
                                  ("1977-1999", slice("1977-01", "1999-12")),
                                  ("2000-2024", slice("2000-01", "2024-12"))]:
        oos_s = oos.loc[sub_slice]; bl_s = bl.loc[sub_slice]
        if len(oos_s) < 25: continue
        out = lw_sharpe_diff_bootstrap(oos_s, bl_s, mean_block_size=12, n_boot=5000, seed=42, alternative="greater")
        rows.append({"protocol": proto, "subperiod": sub_name,
                     "n_months": len(oos_s),
                     "SR_best": out["sr_a"], "SR_base": out["sr_b"],
                     "delta": out["delta"], "p_value": out["p_value"]})
df_sub = pd.DataFrame(rows)
display(df_sub.round(4))
df_sub.to_csv(f"{RESULTS}/03_robustness_subperiod.csv", index=False)


,protocol,subperiod,n_months,SR_best,SR_base,delta,p_value
0,Expanding,Full,596,0.3866,0.3235,0.0631,0.2530
1,Expanding,1977-1999,276,0.4596,0.2458,0.2138,0.0478
2,Expanding,2000-2024,300,0.3063,0.3712,-0.0649,0.6860
3,Rolling 30y,Full,596,0.3634,0.3235,0.0400,0.3218
4,Rolling 30y,1977-1999,276,0.3679,0.2458,0.1221,0.1786
5,Rolling 30y,2000-2024,300,0.3370,0.3712,-0.0342,0.6186


## 3. Robustness over portfolio vol target

In [5]:
rows = []
for pvt in [0.08, 0.10, 0.12]:
    g = grid_backtest(ff49, rf, port_vol_target=pvt, tcost_bps=20.0)
    for proto, tw in [("Expanding", "expanding"), ("Rolling 30y", 30)]:
        oos, _ = walkforward_oos(g, warmup_end="1976-06-30", train_window=tw)
        bl = g.loc[oos.index, "fast=1,wf=+0.0"]
        out = lw_sharpe_diff_bootstrap(oos, bl, mean_block_size=12, n_boot=5000, seed=42, alternative="greater")
        rows.append({"port_vol_target": pvt, "protocol": proto,
                     "SR_best": out["sr_a"], "SR_base": out["sr_b"],
                     "delta": out["delta"], "p_value": out["p_value"]})
df_pvt = pd.DataFrame(rows)
display(df_pvt.round(4))
df_pvt.to_csv(f"{RESULTS}/03_robustness_portvol.csv", index=False)


,port_vol_target,protocol,SR_best,SR_base,delta,p_value
0,0.08,Expanding,0.3866,0.3219,0.0647,0.2486
1,0.08,Rolling 30y,0.3620,0.3219,0.0401,0.3244
2,0.10,Expanding,0.3866,0.3235,0.0631,0.2530
3,0.10,Rolling 30y,0.3634,0.3235,0.0400,0.3218
4,0.12,Expanding,0.3883,0.3237,0.0646,0.2518
5,0.12,Rolling 30y,0.3575,0.3237,0.0338,0.3484


## Summary

The primary null result is robust to:
- Transaction cost variations (0–40 bps)
- Sub-period splits (1977–1999 vs 2000–2024)
- Portfolio vol target (8%–12%)

Re-estimation frequency (quarterly/triennial) requires modifying the walk-
forward step size; deferred to follow-up analysis if warranted.
